In [0]:
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 72.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

os.environ["KAGGLE_USERNAME"] = ""
os.environ["KAGGLE_KEY"] = ""

print("Kaggle credentials configured!")

Kaggle credentials configured!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [00:30<00:00, 149MB/s]


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-750dcb76-2a8c-4ebf-8c4e-83 nogroup 8.4G Jan 14 11:56 2019-Nov.csv
-rwxrwxrwx 1 spark-750dcb76-2a8c-4ebf-8c4e-83 nogroup 5.3G Jan 14 11:58 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 14 11:48 delta
-rwxrwxrwx 1 spark-750dcb76-2a8c-4ebf-8c4e-83 nogroup 4.3G Jan 14 11:56 ecommerce-behavior-data-from-multi-category-store.zip
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 14 11:48 outputs


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 spark-750dcb76-2a8c-4ebf-8c4e-83 nogroup 8.4G Jan 14 11:56 2019-Nov.csv
-rwxrwxrwx 1 spark-750dcb76-2a8c-4ebf-8c4e-83 nogroup 5.3G Jan 14 11:58 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 14 11:48 delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 14 11:48 outputs


In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
csv_path = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/delta/events_oct2019"

db = "workspace.ecommerce"
table_managed = f"{db}.events_oct2019_managed"
table_external = f"{db}.events_oct2019_external"

events = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(csv_path))

events = (events
          .withColumn("price", F.col("price").cast("double")))

In [0]:
(events.write
 .format("delta")
 .mode("overwrite")
 .save(delta_path))

print("Delta written to:", delta_path)

Delta written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/events_oct2019


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {db}")

(events.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(table_managed))

display(spark.table(table_managed).limit(5))

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-10-13T16:33:33.000Z,view,4501525,2053013563877884791,appliances.kitchen.hob,midea,130.54,512773672,e629aaa6-fc0b-478d-bebc-2af6e95bb51c
2019-10-13T16:33:33.000Z,view,1005208,2053013555631882655,electronics.smartphone,apple,424.7,515343194,d44fc41d-bc2a-4f6a-b883-97f2feced972
2019-10-13T16:33:33.000Z,view,4900018,2053013555220840837,appliances.kitchen.juicer,philips,109.5,545204644,1a706b88-1477-4cb2-87ce-0c74e4e2c4cb
2019-10-13T16:33:33.000Z,view,2800605,2053013563835941749,appliances.kitchen.refrigerators,indesit,264.99,517228896,d209cef4-00df-442f-8d60-fe295b254ffd
2019-10-13T16:33:33.000Z,view,1004739,2053013555631882655,electronics.smartphone,xiaomi,187.89,516872406,7f685123-60bb-4e98-9c8a-933c222b8af7


In [0]:
wrong_schema = (events.limit(10)
                .select(
                    "event_time","event_type","product_id","category_id","category_code","brand",
                    F.col("price").cast("string").alias("price"), 
                    "user_id","user_session"
                ))

try:
    (wrong_schema.write
     .format("delta")
     .mode("append")
     .saveAsTable(table_managed))
    print("Unexpected: append succeeded")
except Exception as e:
    print("Schema enforcement triggered (expected).")
    print(str(e)[:600])

Schema enforcement triggered (expected).
[DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'price' and 'price'.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	at com.databricks.sql.transaction.tahoe.schema.SchemaMergingUtils$.$anonfun$mergeDataTypes$1(SchemaMergingUtils.scala:231)
	at scala.collection.ArrayOps$.map$extension(ArrayOps.scala:936)
	at com.databricks.sql.transaction.tahoe.schema.SchemaMergingUtils$.merge$1(SchemaMergingUtils.scala:217)
	at com.databricks.sql.transaction.tahoe.schema.SchemaMergingUtils$.mergeDataTypes(SchemaMergingUtils.scala:335)
	at com.databricks.sql.transaction.tahoe


In [0]:
key_cols = ["user_session", "event_time", "event_type", "product_id"]

events_keyed = (events
    .withColumn("dedupe_key",
                F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in key_cols]), 256))
)

target_keyed = f"{db}.events_oct2019_keyed"
(events_keyed.write.format("delta").mode("overwrite").saveAsTable(target_keyed))

incoming = events_keyed.limit(5000).dropDuplicates(["dedupe_key"])
incoming.createOrReplaceTempView("incoming_events_deduped")

In [0]:
%sql
MERGE INTO workspace.ecommerce.events_oct2019_keyed AS t
USING incoming_events_deduped AS s
ON t.dedupe_key = s.dedupe_key
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
from pyspark.sql import functions as F

db = "workspace.ecommerce"
target = f"{db}.events_oct2019_keyed"
key_cols = ["user_session", "event_time", "event_type", "product_id"]

def with_dedupe_key(df):
    return df.withColumn(
        "dedupe_key",
        F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in key_cols]), 256)
    )

base = spark.table(target).drop("dedupe_key")

updates = (base.limit(2000)
           .withColumn("price", F.col("price") + F.lit(1.0))  
          )

updates = with_dedupe_key(updates).dropDuplicates(["dedupe_key"])
updates.createOrReplaceTempView("updates")

In [0]:
%sql
MERGE INTO workspace.ecommerce.events_oct2019_keyed AS t
USING updates AS s
ON t.dedupe_key = s.dedupe_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2000,2000,0,0


In [0]:
%sql
DESCRIBE HISTORY workspace.ecommerce.events_oct2019_keyed;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-01-14T12:03:37.000Z,74641345985031,ark4nandi@gmail.com,MERGE,"Map(predicate -> [""(dedupe_key#14669 = dedupe_key#14621)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3959910042445916),0114-115536-51th6ynw-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 130708, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2000, executionTimeMs -> 7735, materializeSourceTimeMs -> 887, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3516, numTargetRowsUpdated -> 2000, numOutputRows -> 2000, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1994, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3216)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
3,2026-01-14T12:02:43.000Z,74641345985031,ark4nandi@gmail.com,MERGE,"Map(predicate -> [""(dedupe_key#14078 = dedupe_key#14053)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3959910042445916),0114-115536-51th6ynw-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 8872, materializeSourceTimeMs -> 12, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4997, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 8731)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
2,2026-01-14T12:02:31.000Z,74641345985031,ark4nandi@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3959910042445916),0114-115536-51th6ynw-v2n,1,WriteSerializable,false,"Map(numFiles -> 43, numRemovedFiles -> 43, numRemovedBytes -> 2226999842, numDeletionVectorsRemoved -> 0, numOutputRows -> 42448764, numOutputBytes -> 2226999842)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
1,2026-01-12T20:35:06.000Z,74641345985031,ark4nandi@gmail.com,MERGE,"Map(predicate -> [""(dedupe_key#16035 = dedupe_key#16010)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1180259694811334),0112-175615-yhefu0if-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 6949, materializeSourceTimeMs -> 12, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4997, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTi

In [0]:
v0 = (spark.read.format("delta")
      .option("versionAsOf", 0)
      .table(target))

display(v0.limit(10))

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,dedupe_key
2019-10-27T03:11:09.000Z,view,12711053,2053013553559896355,null,tunga,32.69,538720752,8c12a5f4-a153-1a33-e91d-97f0825a180f,71b0dac87daa999e684f6a03ee415a5f215d4892180448af613b6d3d97e88e8c
2019-10-27T03:11:09.000Z,view,26600074,2053013563517174627,null,lucente,76.96,511397207,475d635b-ab81-4f6f-afb1-6f307a8c2659,7dc653a39b2ee07b6c512826ff83f05be161c80845a7d894f340c022535fcd5f
2019-10-27T03:11:10.000Z,view,28716666,2053013565639492569,apparel.shoes,respect,128.45,529241054,d215f87b-914e-42dc-8651-c2b3adb7d2e9,6b062328056458efaa67546c92c8bab44f11c4f78c8af340edeab5994af63d60
2019-10-27T03:11:10.000Z,view,12720604,2053013553559896355,null,joyroad,35.52,548106688,6923ed1b-e9f6-4c9b-b90b-6dc5be5232d7,6397eadea904753bc320fbe296f072270842e97a5f8306016c4e64800c27cfe3
2019-10-27T03:11:10.000Z,view,1002524,2053013555631882655,electronics.smartphone,apple,531.41,545517399,923915ec-7367-4a13-9cad-453a63ac6fcd,9d169da52af794f61bca590e0553c0904c9f551ab22f49f5787f7e630173fc14
2019-10-27T03:11:10.000Z,view,1307377,2053013558920217191,computers.notebook,lenovo,434.76,513149347,e0d2fbcc-8961-45e7-80ae-2a30f8ca9773,6b294a078831b133bc2b765b8123c42e4ef3785959e9855300f8bffbd79e6be4
2019-10-27T03:11:10.000Z,view,3701190,2053013565983425517,appliances.environment.vacuum,scarlett,123.53,532726458,55db9f87-fab4-418d-9dd7-1dbbf3f9cc5c,0e5a566766a8d66ec73e373b17e771253fb8c62c886124cddf15e7fed52e8f56
2019-10-27T03:11:10.000Z,view,5100375,2053013553341792533,electronics.clocks,xiaomi,70.53,562141510,abde59e7-832a-4835-98f5-63d75f9b1f38,489b72201844a192623b9fd3042bd5b9857eb27a5217a04769f80f62af1f0f8f
2019-10-27T03:11:10.000Z,view,15700211,2053013559733912211,null,null,327.91,544703756,9aa254cd-35b4-4ce9-ad5f-8ccf5abf057c,62ff3675cefcb56ec39471cea087618c5079b60c48f2ae2352aa9944b9015f1b
2019-10-27T03:11:10.000Z,view,29900034,2059484601444729123,null,racer,1207.24,564398376,2a2602a6-e270-43d3-9ed8-c48be8d25d78,8da65566258907f1a7965de302312ffe433722ff9a20832643c5c9ae6a934179


In [0]:
asof = (spark.read.format("delta")
        .option("timestampAsOf", "2026-01-14T12:02:31.000+00:00")
        .table(target))

display(asof.limit(10))

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,dedupe_key
2019-10-24T18:10:26.000Z,view,15100350,2053013557024391671,null,null,321.73,515100947,79daf96d-0dfe-4436-b1e2-6bcf6b1f71ef,a074cc94d8e45ff1c1ba0342d14093c3fa16744191daff6b91a36d095bedca58
2019-10-24T18:10:26.000Z,view,15400069,2070005009256284935,null,intex,74.65,563794339,b9b363e2-5072-4413-a45e-84bf2b4968e5,a8be6b12591d472ee6ce353994575a1b547c70ada729091ea898e4423f01c84c
2019-10-24T18:10:26.000Z,view,7600336,2053013552821698803,null,tp-link,28.06,563067102,9c67e656-a96b-42ff-8556-57162235d8aa,48e76837db816687e46f807211aa5e86d2eb2c81360fb1aac694152914022142
2019-10-24T18:10:26.000Z,view,21408793,2053013561579406073,electronics.clocks,orient,76.19,562654002,234163a8-5891-4c64-bf08-27e928b44f07,40c06c5d3488b78a7940403e73bfe5852b4912d4b13c65ec3e0bb2e22a697044
2019-10-24T18:10:26.000Z,view,22700084,2053013556168753601,null,force,239.32,563794479,ce97919d-bd78-4ccb-9d3f-a0a581697843,e03cfd4466be6a0c6077118c1a4b4104eac09848d85cc5738f7215823da4e4b4
2019-10-24T18:10:26.000Z,view,13101275,2053013553526341921,null,rial,354.74,521263262,787bc26f-f870-425c-b4d2-277130292e8c,3fbdfab52e9722190d3958ebd69548f8931a32194f10afdf9de140e6ef435879
2019-10-24T18:10:26.000Z,view,1005105,2053013555631882655,electronics.smartphone,apple,1397.09,563556164,ff43aaef-6775-435b-847f-4beeb86c4913,4123791555f12ef568eba227219e702c9334241068fb82dad8be5009719bdbcd
2019-10-24T18:10:26.000Z,view,9800218,2053013554071601477,null,air-cool,14.31,562571786,b2cf2f5d-c2fc-44e4-abc0-36933514a920,04a2c4d458b80b0a7a26fd1e0bca7d5f656500ffbecf7b4ec371c9628ee82b50
2019-10-24T18:10:26.000Z,view,1004529,2053013555631882655,electronics.smartphone,samsung,396.15,515221242,3dada809-fb1d-4e8b-9dba-921fcd71bfb6,6d0c7825e53965f671128dc6a76e31b25ca561fb6e3d94870b346e0a4f76874c
2019-10-24T18:10:26.000Z,view,4804294,2053013554658804075,electronics.audio.headphone,meizu,71.82,560583382,d9bb8495-649e-43b2-a105-1118b6cda007,edcc276473d7f6f544e435fc78f7c7002a92c5caa3919401de970682df88ea26


In [0]:
%sql
OPTIMIZE workspace.ecommerce.events_oct2019_keyed
ZORDER BY (event_type, user_id);

path,metrics
,"List(50, 44, List(34565007, 56649615, 4.480241982E7, 50, 2240120991), List(130708, 53347810, 5.061660340909091E7, 44, 2227130550), 0, List(minCubeSize(107374182400), List(0, 0), List(44, 2227130550), 0, List(44, 2227130550), 1, null), null, 0, 1, 44, 0, false, 0, 0, 1768392405989, 1768392432301, 8, 1, null, List(1, 2000), null, 10, 10, 111348, 0, null)"


In [0]:
%sql
VACUUM workspace.ecommerce.events_oct2019_keyed RETAIN 168 HOURS;

path
""
